## Occupancy Data with VC analysis

We have a training set comprised of sensor data and a corresponding classification of whether or not a room is occupied. We will use the pocket algorithm to fit the data and analyze the generalization bounds using the VC dimension.

In [9]:
from __future__ import print_function
import pandas as pd
import numpy as np

import load_data as ld
import pocket as pk
import analysis_tools as at


# setup our training data
training = ld.load_data('datatraining.txt')

# setup our testing data
testing = ld.load_data('datatest.txt')

training.head()

,Temperature,Humidity,Light,CO2,HumidityRatio,Classification
1,23.18,27.2720,426.0,721.25,0.004793,1
2,23.15,27.2675,429.5,714.00,0.004783,1
3,23.15,27.2450,426.0,713.50,0.004779,1
4,23.15,27.2000,426.0,708.25,0.004772,1
5,23.10,27.2000,426.0,704.50,0.004757,1


## The VC_dimension

Before we actually train our classification alogrithm on our dataset, let's take a look at what the VC dimension can tell us about what we're about to do. 

---
**Theorem 2.5** (VC generalization bound).  For any $ \delta > 0,$

$$ E_{out}(g) \le E_{in}(g) + \sqrt{ \frac{8}{N} ln \frac{4mH(2N)}{\delta}} $$

with probability $\ge 1 - \delta $

---

What we are saying is that in order to know how well our algorithm will generalize to data that it has not seen($E_{out}$), we must frame it in terms of $E_{in}$ + the result of our VC calcuation. 

The important part is that our VC calculation is dependent on $N$(amount of data we train our algorithm with), and $\delta$(percentage error we are willing to tolerate). 

As an example, let us say that our $E_{in}$ is *.06* and the result of our VC calculation is *.04*, where our error tolerance $\delta$ was *.03*.

We would then be able to say

$$ E_{out} \le .06 + .04 \ with \ probability \ge 1 - .03 $$

This in effect says with 97% certainty, $E_{out}$ will be less than .10%. So we are 97% certain that our algorithm will missclassify no more than 10% of samples that it has not seen. 

Powerful stuff, but does it work? 


---
## Run 1

Let's train the pocket algorithm with 100 samples and shoot for an error tolerance of 0.07. This will give us

N=100, delta=.07

In [2]:
hundred_samples = training.sample(n=100)

fit = pk.pocket(hundred_samples, iterations=30)
w= fit['w']

e_in = at.misclassified_count(w=w, data=training)
e_out = at.misclassified_count(w=w, data=testing)
vc_bound = at.vc_bound(N=100, tolerance=.07, mH=3)

at.print_results(e_in=e_in, e_out=e_out, vc=vc_bound)



E_in number misclassified: 8143, percentage error: 1.0
E_out number misclassified: 2665, percentage error: 1.0
VC bound: 0.683363087518




As can be seen above, our algorithm failed miserably, and the VC bound isn't helping us much either. We would have essentially 1.68 percent chance that our algorithm will perform badly. Clearly, we will need to make adjustments

---
## Run 2

The natural thing to do at this point is to increase our N, this should improve our performance. Let's  

N=1000, delta=.07

In [5]:
thousand_samples = training.sample(n=1000)

fit = pk.pocket(thousand_samples, iterations=30)
w=fit['w']

e_in = at.misclassified_count(w=w, data=training)
e_out = at.misclassified_count(w=w, data=testing)
vc_bound = at.vc_bound(N=1000, tolerance=.07, mH=3)

at.print_results(e_in=e_in, e_out=e_out, vc=vc_bound)

E_in number missclassified: 8143, percentage error: 1.0
E_out number missclassified: 2665, percentage error: 1.0
VC bound: 0.216098382544


Here we see that again our algorithm performed very badly in sample, but the VC bound is much better, however that is of little help to us when our in sample error is so high. 

---
## Run 3

We seem to be having some problems doing any kind of generalization well, what could be the issue? 

Let's take a look at our data set and see what the ratio is for the classes 

In [4]:
class_count  = training['Classification'].value_counts()
print class_count

-1    6414
 1    1729
Name: Classification, dtype: int64


Hmm, so the ratio of classes is very uneven. So let's see what that cound looks like in the case of our first run, where we just sampled 100 random points from the training data.  

In [5]:
hundred_samples = training.sample(n=100)
print hundred_samples['Classification'].value_counts()

-1    81
 1    19
Name: Classification, dtype: int64


Looking at these counts, we can roughly see that the *occupied* class makes up only 23-26% of the samples. This will give the pocket algorithm problems as it will continually see 'outliers', making the fit troublesome.  

There are two ways to remedy the issue

1) Give it more and more data. At around 5000 samples for this data set we will get good convergence

**Golen rule of machine learning**

> data conquers all. 

**Silver rule**

> golden rule applies, assuming you have the computational horsepower to train on all that data

**corollary**
> often you don't, but you do the best you can:)

2) Choose your samples more carefully.  This is dangerous, and we must still randomly sample, but do so more intelligently. 

---
## Run 4

This time we will use 100 samples as in our first run, but this time we will choose 100 samples randomly, but this
time ensuring that we get 50 samples with classification *1*, and 50 with *-1*. We will see if this improves our in sample error. 

In [3]:
hundred_samples = pd.concat([training[training['Classification']==1].sample(n=50),
                      training[training['Classification']==-1].sample(n=50)],
                      axis=0).reset_index(drop=True)

fit = pk.pocket(hundred_samples, iterations=30)
w = fit['w']
e_in = at.misclassified_count(w=w, data=training)
e_out  = at.misclassified_count(w=w, data=testing)
vc_bound = at.vc_bound(N=100, tolerance=.07, mH=3)

at.print_results(e_in=e_in, e_out=e_out, vc=vc_bound)


E_in number missclassified: 670, percentage error: 0.0822792582586
E_out number missclassified: 205, percentage error: 0.0769230769231
VC bound: 0.683363087518


Well....not bad right?  We have achieved an error rate of 0.073 with just 50 examples from each class. However the generalization bound is telling us we have 93% chance of doing better than .75% missclassified. Seems rough right? 

---
## Run 5

We can now try appoach 2 where we increase N. We might as well give it all the samples and then examine the classification error

N=8143, delta=.07

In [2]:

fit = pk.pocket(training, iterations=30)
w = fit['w']

e_in = at.misclassified_count(w=w, data=training)
e_out = at.misclassified_count(w=w, data=testing)

vc_bound = at.vc_bound(N=len(training), tolerance=.07, mH=3)

at.print_results(e_in=e_in, e_out=e_out, vc=vc_bound)


E_in number missclassified: 558, percentage error: 0.0685251135945
E_out number missclassified: 184, percentage error: 0.06904315197
VC bound: 0.0757284902891
